In [ ]:
pip install ucimlrepo

In [ ]:
# Importando as bibliotecas necessárias
from tensorflow.keras.layers import Input, Dense
from tensorflow.keras.models import Model
import pandas as pd
from sklearn.preprocessing import LabelEncoder, MinMaxScaler
from tensorflow import keras
from tensorflow.keras.layers import Input, Dense
from tensorflow.keras.models import Model
import numpy as np

In [ ]:
from ucimlrepo import fetch_ucirepo

# fetch dataset
online_retail = fetch_ucirepo(id=352)

# data (as pandas dataframes)
X = online_retail.data.features
y = online_retail.data.targets

# metadata
print(online_retail.metadata)

# variable information
print(online_retail.variables)

{'uci_id': 352, 'name': 'Online Retail', 'repository_url': 'https://archive.ics.uci.edu/dataset/352/online+retail', 'data_url': 'https://archive.ics.uci.edu/static/public/352/data.csv', 'abstract': 'This is a transactional data set which contains all the transactions occurring between 01/12/2010 and 09/12/2011 for a UK-based and registered non-store online retail.', 'area': 'Business', 'tasks': ['Classification', 'Clustering'], 'characteristics': ['Multivariate', 'Sequential', 'Time-Series'], 'num_instances': 541909, 'num_features': 6, 'feature_types': ['Integer', 'Real'], 'demographics': [], 'target_col': None, 'index_col': ['InvoiceNo', 'StockCode'], 'has_missing_values': 'no', 'missing_values_symbol': None, 'year_of_dataset_creation': 2015, 'last_updated': 'Mon Oct 21 2024', 'dataset_doi': '10.24432/C5BW33', 'creators': ['Daqing Chen'], 'intro_paper': {'ID': 361, 'type': 'NATIVE', 'title': 'Data mining for the online retail industry: A case study of RFM model-based customer segmenta

In [ ]:
# Retirando a coluna de Data e Hora da compra
X = X.drop(columns=['InvoiceDate'])

In [ ]:
# Channel Islands não possui DDI próprio então foi colocado o número 4
# Canadá possui o mesmo DDI que USA então foi alterado pra 11
# European Community não possui DDI próprio então foi colocado o número 3

country_mapping = {
    'United Kingdom': 44,
    'Germany': 49,
    'France': 33,
    'EIRE': 353,
    'Spain': 34,
    'Netherlands': 31,
    'Belgium': 32,
    'Switzerland': 41,
    'Portugal': 351,
    'Australia': 61,
    'Norway': 47,
    'Italy': 39,
    'Channel Islands': 4,
    'Finland': 358,
    'Cyprus': 657,
    'Sweden': 46,
    'Unspecified': 0,
    'Austria': 43,
    'Denmark': 45,
    'Japan': 81,
    'Poland': 48,
    'Israel': 972,
    'USA': 1,
    'Hong Kong': 852,
    'Singapore': 65,
    'Iceland': 354,
    'Canada': 11,
    'Greece': 30,
    'Malta': 356,
    'United Arab Emirates': 971,
    'European Community': 3,
    'RSA': 27,
    'Lebanon': 961,
    'Lithuania': 370,
    'Brazil': 55,
    'Czech Republic': 420,
    'Bahrain': 973,
    'Saudi Arabia': 966
}

X['Country'] = X['Country'].map(country_mapping).astype(int)

In [ ]:
X_sample = X.sample(frac=0.5, random_state=42)  # Seleciona 50% dos dados
X['Quantity'] = X['Quantity'].astype('int32')
X['Country'] = X['Country'].astype('int16')


# Usar pd.get_dummies para transformar Description em colunas de 0 e 1
df_encoded = pd.get_dummies(X_sample, columns=['Description'], prefix='', prefix_sep='').fillna(0).astype(int)

# Ver resultado
print(df_encoded.head())

        Quantity  UnitPrice  CustomerID  Country  \
209268        24          0       17315       44   
207108         4          6       14031       44   
167085         4          0       14031       44   
471836         3          1       17198       44   
115865         2          9       13502       44   

         4 PURPLE FLOCK DINNER CANDLES   50'S CHRISTMAS GIFT BAG LARGE  \
209268                               0                               0   
207108                               0                               0   
167085                               0                               0   
471836                               0                               0   
115865                               0                               0   

         DOLLY GIRL BEAKER   I LOVE LONDON MINI BACKPACK  \
209268                   0                             0   
207108                   0                             0   
167085                   0                             0   
47

In [ ]:
print(X_sample['Country'].value_counts())

Country
44     247795
49       4760
33       4275
353      4145
34       1270
31       1175
32       1035
41        935
351       768
61        618
47        540
39        394
4         378
358       329
657       297
46        237
0         217
43        215
45        189
81        184
48        166
1         159
972       147
852       146
65        112
354        95
30         72
11         64
356        56
971        44
27         28
3          26
370        19
961        18
55         17
420        15
973         9
966         5
Name: count, dtype: int64


In [ ]:
X_sample.info()

<class 'pandas.core.frame.DataFrame'>
Index: 270954 entries, 209268 to 219400
Data columns (total 5 columns):
 #   Column       Non-Null Count   Dtype  
---  ------       --------------   -----  
 0   Description  270240 non-null  object 
 1   Quantity     270954 non-null  int64  
 2   UnitPrice    270954 non-null  float64
 3   CustomerID   203319 non-null  float64
 4   Country      270954 non-null  int64  
dtypes: float64(2), int64(2), object(1)
memory usage: 12.4+ MB


In [ ]:
print(X_sample.describe())

            Quantity      UnitPrice     CustomerID        Country
count  270954.000000  270954.000000  203319.000000  270954.000000
mean        9.244030       4.594357   15290.406273      51.762853
std       260.164411      80.433597    1713.240410      57.778485
min    -80995.000000  -11062.060000   12346.000000       0.000000
25%         1.000000       1.250000   13956.000000      44.000000
50%         3.000000       2.080000   15152.000000      44.000000
75%        10.000000       4.130000   16794.000000      44.000000
max     74215.000000   17836.460000   18287.000000     973.000000


In [ ]:
numerical_cols = X_sample.select_dtypes(include=['number']).columns
scaler = MinMaxScaler()
X_sample[numerical_cols] = scaler.fit_transform(X_sample[numerical_cols])

In [ ]:
string_cols = X_sample.select_dtypes(include=['object']).columns
numerical_cols = X.select_dtypes(include=['number']).columns
X_numerical = X[numerical_cols]
print(X.isnull().sum())  # Check for NaN values


label_encoders = {}
for col in string_cols:
    label_encoders[col] = LabelEncoder()
    X_sample[col] = label_encoders[col].fit_transform(X_sample[col])

Description      1454
Quantity            0
UnitPrice           0
CustomerID     135080
Country             0
dtype: int64


In [ ]:
input_dim = X_sample.shape[1]  # 4 itens de entrada no dataset

In [ ]:
from tensorflow import keras
# Construir o Autoencoder
input_layer = Input(shape=(input_dim,))
encoder = Dense(3, activation="relu")(input_layer)  # Consider using 'tanh' or 'linear'
decoder = Dense(input_dim, activation="linear")(encoder) # Use 'linear' for regression
autoencoder = Model(inputs=input_layer, outputs=decoder)

In [ ]:
# Definir o modelo
autoencoder.compile(optimizer='adam', loss='mse')  # Use 'mse' for regression

In [ ]:
# # Compilar o modelo
# autoencoder.fit(X_sample, X_sample, epochs=100, verbose=0)

autoencoder.fit(X_sample, X_sample, epochs=30, batch_size=64, verbose=0)
# Utilizando a Sample

In [ ]:
# Fazer previsões (reconstrução das entradas)
reconstructed = autoencoder.predict(X_sample)
print(reconstructed)

8468/8468 ━━━━━━━━━━━━━━━━━━━━ 9s 1ms/step
[[nan nan nan nan nan]
 [nan nan nan nan nan]
 [nan nan nan nan nan]
 ...
 [nan nan nan nan nan]
 [nan nan nan nan nan]
 [nan nan nan nan nan]]


In [ ]:
# Identificar colunas não numéricas
non_numeric_cols = X.select_dtypes(include=['object']).columns

# Aplicar Label Encoding ou One-Hot Encoding nas colunas categóricas
for col in non_numeric_cols:
    le = LabelEncoder()
    X[col] = le.fit_transform(X[col])

# Verifique o resultado
print(X.head())

   Description  Quantity  UnitPrice  CustomerID  Country
0         3918         6       2.55     17850.0       44
1         3926         6       3.39     17850.0       44
2          913         8       2.75     17850.0       44
3         1910         6       3.39     17850.0       44
4         2911         6       3.39     17850.0       44


In [ ]:
numerical_cols = X.select_dtypes(include=['number']).columns  # Get numerical columns from X
scaler = MinMaxScaler()  # Create a new or reset the existing scaler
X[numerical_cols] = scaler.fit_transform(X[numerical_cols]) # Fit and transform on X

In [ ]:
# Fazer previsões (reconstrução das entradas)
# Após o treinamento, o autoencoder tenta reconstruir as amostras de entrada
reconstructed = autoencoder.predict(X)
print("Dados originais:")
print(X_sample)
print("\nDados reconstruídos:")
print(reconstructed)  # Exibe os dados reconstruídos, que devem ser próximos aos dados originais

16935/16935 ━━━━━━━━━━━━━━━━━━━━ 19s 1ms/step
Dados originais:
        Description  Quantity  UnitPrice  CustomerID   Country
209268         1607  0.521996   0.382819    0.836391  0.045221
207108         1486  0.521867   0.383030    0.283622  0.045221
167085         3212  0.521867   0.382812    0.283622  0.045221
471836         2250  0.521861   0.382857    0.816698  0.045221
115865         2838  0.521854   0.383134    0.194580  0.045221
...             ...       ...        ...         ...       ...
284267         3907  0.521848   0.382848    0.751894  0.045221
91106          1605  0.521874   0.382819    0.872580  0.045221
92813          3087  0.521848   0.382920         NaN  0.045221
211454         2602  0.521861   0.382847    0.028783  0.050360
219400         2423  0.521874   0.383163         NaN  0.045221

[270954 rows x 5 columns]

Dados reconstruídos:
[[nan nan nan nan nan]
 [nan nan nan nan nan]
 [nan nan nan nan nan]
 ...
 [nan nan nan nan nan]
 [nan nan nan nan nan]
 [nan nan na

In [ ]:
# import pandas as pd
# from sklearn.preprocessing import LabelEncoder

# # Assuming X is a pandas DataFrame
# # ... your existing code for creating the autoencoder model ...

# # 1. Identify string columns in X
# string_cols = X.select_dtypes(include=['object']).columns

# # 2. Create a LabelEncoder for each string column
# label_encoders = {}
# for col in string_cols:
#     label_encoders[col] = LabelEncoder()
#     X[col] = label_encoders[col].fit_transform(X[col])

# # 3. Now you can train the model
# autoencoder.fit(X, X, epochs=100, verbose=0)